<a href="https://colab.research.google.com/github/herrypu/HP_LLM4CD_S6SP/blob/main/chipchat_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM4ChipDesign HW1 Chipchat
## Extension (Parameterizing Example 2: Sequence Detector)


In [1]:
API_KEY="sk-4b4b50b8153147759b5530268e51a1f7"
MAX_TOKENS=4068
TEMPERATURE=0.7

### 1. Setting up

In [2]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [3]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = API_KEY,
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )
    content = completion.choices[0].message.content

    print("-" * 30 + " LLM Response " + "-" * 30)
    print(content)
    print("-" * 74)

    return content

### 2. Verilog Generation

In [4]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   4749      0 --:--:-- --:--:-- --:--:--  4761


In [5]:
verilog_generation_prompt = '''
I want to extend the sequence_detector design to be fully parameterized.
The new module should be named "sequence_detector_ext".

Specifications:
1. Parameters:
   - DATA_WIDTH: Default is 3 (width of input data).
   - SEQ_LEN: Default is 8 (number of steps in the sequence).
   - TARGET_SEQ: A parameter that holds the entire concatenated sequence.
     Default should be the concatenation of: 0b001, 101, 110, 000, 110, 110, 011, 101.

2. Logic Requirements:
   - Port Names: clk, reset_n, data [DATA_WIDTH-1:0], sequence_found.
   - Design Style: Use a shift-register window approach OR a generic FSM that references TARGET_SEQ.
   - Timing: Must be a Mealy-type output (combinational) that asserts "sequence_found"
     in the SAME cycle the last element of the sequence is matched.

3. Synthesizability:
   - Must use synthesizable Verilog (always @(posedge clk) and always @(*)).
'''

In [6]:
ask_deepseek(verilog_generation_prompt)

------------------------------ LLM Response ------------------------------
Here's a fully parameterized sequence detector using the shift-register window approach:

```verilog
module sequence_detector_ext #(
    parameter DATA_WIDTH = 3,
    parameter SEQ_LEN = 8,
    parameter TARGET_SEQ = {3'b001, 3'b101, 3'b110, 3'b000, 3'b110, 3'b110, 3'b011, 3'b101}
) (
    input wire clk,
    input wire reset_n,
    input wire [DATA_WIDTH-1:0] data,
    output reg sequence_found
);

    // Internal shift register to hold previous SEQ_LEN-1 values plus current input
    reg [DATA_WIDTH-1:0] shift_reg [0:SEQ_LEN-1];
    
    // Concatenated window for comparison
    wire [DATA_WIDTH*SEQ_LEN-1:0] current_window;
    
    // Initialize concatenated window
    genvar i;
    generate
        for (i = 0; i < SEQ_LEN; i = i + 1) begin : window_concat
            assign current_window[(i+1)*DATA_WIDTH-1:i*DATA_WIDTH] = shift_reg[i];
        end
    endgenerate

    // Shift register update logic
    alway

"Here's a fully parameterized sequence detector using the shift-register window approach:\n\n```verilog\nmodule sequence_detector_ext #(\n    parameter DATA_WIDTH = 3,\n    parameter SEQ_LEN = 8,\n    parameter TARGET_SEQ = {3'b001, 3'b101, 3'b110, 3'b000, 3'b110, 3'b110, 3'b011, 3'b101}\n) (\n    input wire clk,\n    input wire reset_n,\n    input wire [DATA_WIDTH-1:0] data,\n    output reg sequence_found\n);\n\n    // Internal shift register to hold previous SEQ_LEN-1 values plus current input\n    reg [DATA_WIDTH-1:0] shift_reg [0:SEQ_LEN-1];\n    \n    // Concatenated window for comparison\n    wire [DATA_WIDTH*SEQ_LEN-1:0] current_window;\n    \n    // Initialize concatenated window\n    genvar i;\n    generate\n        for (i = 0; i < SEQ_LEN; i = i + 1) begin : window_concat\n            assign current_window[(i+1)*DATA_WIDTH-1:i*DATA_WIDTH] = shift_reg[i];\n        end\n    endgenerate\n\n    // Shift register update logic\n    always @(posedge clk or negedge reset_n) begin\n  

### 3. Extraction and Verification

#### Testbench

In [41]:
tb_code_v3_fixed = '''
`timescale 1ns/1ps

module tb_sequence_detector_ext();

// --- 1. 参数与信号定义 ---
localparam DATA_WIDTH_1 = 3;
localparam SEQ_LEN_1 = 8;
localparam TARGET_SEQ_1 = {3'b101, 3'b011, 3'b110, 3'b110, 3'b000, 3'b110, 3'b101, 3'b001};

localparam DATA_WIDTH_2 = 4;
localparam SEQ_LEN_2 = 5;
localparam TARGET_SEQ_2 = {4'hE, 4'hD, 4'hC, 4'hB, 4'hA};

localparam DATA_WIDTH_3 = 2;
localparam SEQ_LEN_3 = 3;
localparam TARGET_SEQ_3 = {2'b11, 2'b10, 2'b01};

reg clk;
reg reset_n;
reg [DATA_WIDTH_1-1:0] data_1;
reg [DATA_WIDTH_2-1:0] data_2;
reg [DATA_WIDTH_3-1:0] data_3;
wire sequence_found_1, sequence_found_2, sequence_found_3;

// --- 2. 实例化 DUT ---
sequence_detector_ext #(.DATA_WIDTH(DATA_WIDTH_1), .SEQ_LEN(SEQ_LEN_1), .TARGET_SEQ(TARGET_SEQ_1)) dut_1 (.clk(clk), .reset_n(reset_n), .data(data_1), .sequence_found(sequence_found_1));
sequence_detector_ext #(.DATA_WIDTH(DATA_WIDTH_2), .SEQ_LEN(SEQ_LEN_2), .TARGET_SEQ(TARGET_SEQ_2)) dut_2 (.clk(clk), .reset_n(reset_n), .data(data_2), .sequence_found(sequence_found_2));
sequence_detector_ext #(.DATA_WIDTH(DATA_WIDTH_3), .SEQ_LEN(SEQ_LEN_3), .TARGET_SEQ(TARGET_SEQ_3)) dut_3 (.clk(clk), .reset_n(reset_n), .data(data_3), .sequence_found(sequence_found_3));

// --- 3. 时钟生成 ---
always #5 clk = ~clk;

// --- 4. 任务定义 (必须在 endmodule 之前！) ---
task send_sequence_1;
    input integer pattern_num;
    begin
        if (pattern_num == 1) begin
            @(posedge clk); data_1 = 3'b001;
            @(posedge clk); data_1 = 3'b101;
            @(posedge clk); data_1 = 3'b110;
            @(posedge clk); data_1 = 3'b000;
            @(posedge clk); data_1 = 3'b110;
            @(posedge clk); data_1 = 3'b110;
            @(posedge clk); data_1 = 3'b011;
            @(posedge clk); data_1 = 3'b101;
        end
    end
endtask

task send_sequence_2;
    begin
        @(posedge clk); data_2 = 4'hA;
        @(posedge clk); data_2 = 4'hB;
        @(posedge clk); data_2 = 4'hC;
        @(posedge clk); data_2 = 4'hD;
        @(posedge clk); data_2 = 4'hE;
    end
endtask

task send_sequence_3;
    begin
        @(posedge clk); data_3 = 2'b01;
        @(posedge clk); data_3 = 2'b10;
        @(posedge clk); data_3 = 2'b11;
    end
endtask

// --- 5. 测试流程 ---
initial begin
    clk = 0; reset_n = 0; data_1 = 0; data_2 = 0; data_3 = 0;
    #20 reset_n = 1;

    $display("Test 1: Default Pattern");
    send_sequence_1(1);
    #2; if (sequence_found_1) $display("  PASS"); else $display("  FAIL");

    $display("Test 2: 4-bit Hex");
    send_sequence_2();
    #2; if (sequence_found_2) $display("  PASS"); else $display("  FAIL");

    $display("Test 3: Short Seq");
    send_sequence_3();
    #2; if (sequence_found_3) $display("  PASS"); else $display("  FAIL");

    #100 $finish;
end

endmodule // 模块在最后结束
'''

with open("sequence_detector/sequence_detector_ext_tb.v", "w") as f:
    f.write(tb_code_v3_fixed)

#### First Approach Verilog Verification

In [42]:
output_verilog_code = '''
module sequence_detector_ext #(
    parameter DATA_WIDTH = 3,
    parameter SEQ_LEN = 8,
    parameter TARGET_SEQ = {3'b001, 3'b101, 3'b110, 3'b000, 3'b110, 3'b110, 3'b011, 3'b101}
) (
    input wire clk,
    input wire reset_n,
    input wire [DATA_WIDTH-1:0] data,
    output reg sequence_found
);

    // Internal shift register to hold previous SEQ_LEN-1 values plus current input
    reg [DATA_WIDTH-1:0] shift_reg [0:SEQ_LEN-1];

    // Concatenated window for comparison
    wire [DATA_WIDTH*SEQ_LEN-1:0] current_window;

    // Initialize concatenated window
    genvar i;
    generate
        for (i = 0; i < SEQ_LEN; i = i + 1) begin : window_concat
            assign current_window[(i+1)*DATA_WIDTH-1:i*DATA_WIDTH] = shift_reg[i];
        end
    endgenerate

    // Shift register update logic
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            // Reset all shift register elements to 0
            for (int j = 0; j < SEQ_LEN; j = j + 1) begin
                shift_reg[j] <= {DATA_WIDTH{1'b0}};
            end
        end
        else begin
            // Shift data through the register
            shift_reg[0] <= data;
            for (int j = 1; j < SEQ_LEN; j = j + 1) begin
                shift_reg[j] <= shift_reg[j-1];
            end
        end
    end

    // Mealy output logic (combinational)
    always @(*) begin
        sequence_found = (current_window == TARGET_SEQ) ? 1'b1 : 1'b0;
    end

endmodule
'''
filename = "sequence_detector/sequence_detector_ext.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [43]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector_ext.vvp sequence_detector_ext.v sequence_detector_ext_tb.v && vvp sequence_detector_ext.vvp

Test 1: Default Pattern
  FAIL
Test 2: 4-bit Hex
  FAIL
Test 3: Short Seq
  FAIL


#### Second Approach Verilog Verification

In [44]:
output_verilog_code = '''
module sequence_detector_ext #(
    parameter DATA_WIDTH = 3,
    parameter SEQ_LEN = 8,
    parameter TARGET_SEQ = {3'b001, 3'b101, 3'b110, 3'b000, 3'b110, 3'b110, 3'b011, 3'b101}
) (
    input wire clk,
    input wire reset_n,
    input wire [DATA_WIDTH-1:0] data,
    output reg sequence_found
);

    // State encoding using binary counter
    reg [$clog2(SEQ_LEN)-1:0] state, next_state;

    // Extract sequence elements for comparison
    wire [DATA_WIDTH-1:0] seq_elements [0:SEQ_LEN-1];

    generate
        for (genvar i = 0; i < SEQ_LEN; i = i + 1) begin : seq_extract
            assign seq_elements[i] = TARGET_SEQ[(i+1)*DATA_WIDTH-1:i*DATA_WIDTH];
        end
    endgenerate

    // State transition logic (combinational)
    always @(*) begin
        next_state = state;
        sequence_found = 1'b0;

        if (state == SEQ_LEN-1 && data == seq_elements[SEQ_LEN-1]) begin
            next_state = 0; // Reset to initial state after match
            sequence_found = 1'b1;
        end
        else if (data == seq_elements[state]) begin
            next_state = state + 1; // Move to next state
        end
        else begin
            next_state = 0; // Reset on mismatch
        end
    end

    // State register
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= 0;
        end
        else begin
            state <= next_state;
        end
    end

endmodule
'''
filename = "sequence_detector/sequence_detector_ext.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [45]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector_ext.vvp sequence_detector_ext.v sequence_detector_ext_tb.v && vvp sequence_detector_ext.vvp

Test 1: Default Pattern
  PASS
Test 2: 4-bit Hex
  PASS
Test 3: Short Seq
  PASS
